In [ ]:
import ginsim 
import biolqm
import boolsim
import pandas as pd
from colomoto_jupyter import tabulate
from colomoto.minibn import BooleanNetwork
from colomoto.temporal_logics import *

## boolsim

In [ ]:
## Define network
network = {
    "ATM": "!Wip1",
    "Chk2": "ATM & !Wip1",
    "Mdm2": "p53",
    "p53": "(ATM | Chk2) & !Mdm2",
    "Wip1": "p53"
}
# network = { 
#     "x" : "x|y", 
#     "y" : "y"}
# network = { 
#     "x" : "!z",
#     "y" : "!x",
#     "z" : "!y"}
bn = BooleanNetwork(network)

In [3]:
## Tabulate the attractors using boolsim
%time A = boolsim.attractors(bn)
tabulate(A)

CPU times: user 19.7 ms, sys: 7.02 ms, total: 26.8 ms
Wall time: 126 ms


,x,y
1,0,0
2,1,0
0,1,1


In [4]:
## Print attractors. The asterisk means "complex" attractors 
for att in A:
    print(att)

{'x': 1, 'y': 1}
{'x': 0, 'y': 0}
{'x': 1, 'y': 0}


# bioLQM 

In [5]:
## Convert from boolsim to biolqm
lqm = bn.to_biolqm()

In [6]:
## Save for future reference
biolqm.save(lqm, "network.net", "boolsim")

'network.net'

In [7]:
# Influence graph from biolqm
biolqm.influence_graph(lqm)

# computing graph layout...


### Identification of stable states (fixed points)

In [8]:
fps = biolqm.fixpoints(lqm)
pd.DataFrame(fps)

,x,y
0,0,0
1,1,0
2,1,1


### Identification of stable motifs (trapspaces)

A stable motif (also called symbolic steady state) is a partially assigned state such that all possible successors of all states which belong to the motif also belong to the motif. 

In [9]:
traps = biolqm.trapspace(lqm)
pd.DataFrame(traps)

,x,y
0,0,0
1,1,1
2,1,0


### Stable states

In [10]:
states = biolqm.stable(lqm)
print(states)

[{'x': 0, 'y': 0}, {'x': 1, 'y': 0}, {'x': 1, 'y': 1}]


There are some funcionalities regarding "simulation" of deterministic paths (and non-deterministic by doing random walks).

### Model perturbation

the biolqm.perturbation function enables the construction of a variant of the model, where the logical function of one (or several) component has been modified. A textual parameter describes the modification:

component%0 defines a knockout of a component

component%1 defines an ectopic expression

component%1:2 restricts the range of values for multi-valued components
regulator:component%0 allows to remove a regulator

In [11]:
pert = biolqm.perturbation(lqm, "ATM%1")
# pert = biolqm.perturbation(lqm, "y%1")

In [12]:
fps = biolqm.fixpoints(pert)
pd.DataFrame(fps)

,x,y
0,1,1


In [13]:
traps = biolqm.trapspace(pert, "terminal")
pd.DataFrame(traps)

,x,y
0,1,1


# GINsim

In [14]:
## Convert from biolqm to ginsim
lrg = biolqm.to_ginsim(lqm)
ginsim.show(lrg)

In [15]:
## Fixed points
fps = biolqm.fixpoints(lqm)
print(len(fps), "fixpoints")

3 fixpoints


In [16]:
# First fixed point
ginsim.show(lrg, fps[0])

In [17]:
# Second fixed point
ginsim.show(lrg, fps[1])

In [21]:
ginsim.show(lrg, fps[2])